In [31]:
#imports
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import importlib
import sys
import os
from astropy.cosmology import Planck18 as cosmo
import ast




In [33]:
# --- User toggle ---
Cristina = True
Shar = not Cristina 

# --- System Configurations ---

if Cristina:
    print("[CONFIG] Using Cristina's local MacBook setup")
    sys.path.insert(0, "/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub")
    os.environ["RUBIN_SIM_DATA_DIR"] = "/Users/andradenebula/rubin_sim_data"
    db_dir = "/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub"

elif Shar:
    print("[CONFIG] Using Shar's Dirac server setup")
    sys.path.insert(0, "/lustre/lrspec/metrics")
    sys.path.insert(0, "/home/3155/metrics/Multi_Transient_Metrics_Hub")
    os.environ["RUBIN_SIM_DATA_DIR"] = "/lustre/lrspec/metrics/rubin_sim_data"
    db_dir = "/lustre/lrspec/metrics"

# Shared config
sys.path.append(os.path.abspath(".."))  # For shared_utils


[CONFIG] Using Cristina's local MacBook setup


In [35]:

import os
print(os.getcwd())


/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub/LFBOT


In [37]:
#this happens twice because idk why but it only works like that
#...you still have to run it twice if it gives warnings
# C: Make sure you save your metric file before running. 

metric_filename = "local_LFBOTmetric"
s_u = "shared_utils"

# --- Reload metric module ---
if metric_filename in sys.modules:
    del sys.modules[metric_filename]
metric = __import__(metric_filename)
importlib.reload(metric)

# --- Reload shared_utils module ---
if s_u in sys.modules:
    del sys.modules[s_u]
shared_utils = __import__(s_u)
importlib.reload(shared_utils)

print(f"[INFO] Loaded metric module: {metric_filename}")
print(f"[INFO] Loaded shared_utils module")



[INFO] Loaded metric module: local_LFBOTmetric
[INFO] Loaded shared_utils module


In [39]:
#metric configurations

#control whether we generate new files
generate_new_templates = True
generate_new_pop = True
make_debug_plots = False #toggle whether or not the pop generation makes plots

#population variables
rate_density = 1e-8

# apply None for non-use case 
z_min, z_max = 0.00226, 0.3
dmin, dmax = None, None

#other
gal_lat_cut = None #latitude cut, for Galactic phenomena
use_extinction = True
t_start = 1 #start time in days
t_end = 3652

# Whether to remove Metric_temp_* folders after running
clean_temp = True  # <- NEW toggle

#cadence variables
cadences = ['four_roll_v4.3.1_10yrs', 'baseline_v4.3.1_10yrs']
ignore_triples = False #turn this to true to ignore triples
filters = ['u', 'g', 'r', 'i', 'z', 'y'] #doesn't work rn i think but
#if we wanted to look at less filters then we would adjust that here

# Standardized output paths for this science case
paths = metric.get_output_paths(case_label="LFBOT")  # <- can change to 'KNe' etc.

storage_dir = paths['storage_dir']
templates_file = paths['templates_file']
pop_file = paths['pop_file']


In [41]:
#load and/or generate light curves
shared_lc_model = metric.load_or_generate_templates(
    templates_file=templates_file,
    generate_new=generate_new_templates
)

[INFO] Generating 1000 light curve templates.


TypeError: 'NoneType' object is not iterable

In [ ]:
#plot light curves from pkl file if desired
shared_utils.plot_some_lcs_from_pkl(templates_file, num=3)

In [ ]:
# Load or generate population slicer
slicer = metric.load_or_generate_population(
    t_start=t_start,
    t_end=t_end,
    d_min=dmin,
    d_max=dmax,
    z_min=z_min,
    z_max=z_max,
    seed=42,
    num_lightcurves=1000,
    gal_lat_cut=gal_lat_cut,
    rate_density=rate_density,
    pop_file=pop_file,
    generate_new=generate_new_pop,
    make_debug_plots=make_debug_plots
)


## All 10 years

In [ ]:
#run detection metric
df_obs_arr = shared_utils.run_detect(metric, slicer, cadences, shared_lc_model, db_dir, storage_dir, debug=True, plot=True, clean_temp=clean_temp, use_extinction=use_extinction)

In [ ]:
#choose what to run in run_multi_metrics
# we can remove historical if we want
multi_metrics = metric.get_multi_metrics(shared_lc_model, include=['detect', 'characterize', 'spec_trigger'], use_extinction=use_extinction)


In [ ]:
shared_utils.run_multi_metrics(multi_metrics, slicer, cadences, shared_lc_model, db_dir, storage_dir, ignore_triples=False, plot=True, clean_temp=clean_temp, use_extinction=use_extinction)

In [ ]:

# Combine obs_records from your metric(s)
all_records = []
for m in multi_metrics:
    if hasattr(m, 'obs_records'):
        all_records.extend(m.obs_records.values())

# Convert to DataFrame
obs_data = pd.DataFrame(all_records)
detected_df = obs_data[obs_data['detected'] == True]

detected_df = obs_data[obs_data['detected'] == True]
theta_rad = detected_df['theta_obs']
theta_deg = np.degrees(theta_rad)

plt.hist(theta_deg, bins=30, color='steelblue', edgecolor='black')
plt.xlabel('Inclination angle (deg)')
plt.ylabel('Number of detected LFBOTs')
plt.title('Detected LFBOTs Inclination Angle Distribution')
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(detected_df['peak_mag'], bins=30, edgecolor='black', color='red')
plt.xlabel("Apparent Peak Magnitude")
plt.ylabel("Number of Detected LFBOTs")
plt.title("Distribution of Apparent Peak Magnitudes (Detected LFBOTs)")
plt.gca().invert_xaxis()  # optional: brightest on the left
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:


# Parse the observation strings back into arrays
def to_array(col):
    return col.apply(lambda x: np.array(ast.literal_eval(x)) if isinstance(x, str) else np.array(x) if isinstance(x, list) else np.array([]))

# Extract magnitudes where SNR >= 5
def get_detected_mags(mags, snrs):
    try:
        mags = np.array(mags)
        snrs = np.array(snrs)
        if len(mags) != len(snrs):
            return []
        return mags[snrs >= 5.0]
    except Exception as e:
        print(f"[DEBUG] Issue with mags/snrs: {e}")
        return []

detected_df = obs_data[obs_data['detected'] == True].copy()  # Use .copy() here

detected_df.loc[:, 'mag_obs'] = to_array(detected_df['mag_obs'])
detected_df.loc[:, 'snr_obs'] = to_array(detected_df['snr_obs'])

detected_df.loc[:, 'det_mags'] = detected_df.apply(
    lambda row: get_detected_mags(row['mag_obs'], row['snr_obs']), axis=1
)

all_detected_mags = np.concatenate(detected_df['det_mags'].values)

plt.figure(figsize=(8, 4))
plt.hist(all_detected_mags, bins=30, edgecolor='black', color='darkorange')
plt.xlabel("Magnitude at Detection (SNR ≥ 5)")
plt.ylabel("Number of Observations")
plt.title("Distribution of Magnitudes at Detection (LFBOTs)")
plt.gca().invert_xaxis()  # brighter on the left
plt.grid(True)
plt.tight_layout()
plt.show()

